# Unit 4 — companion notebook
Code for the worked examples in the [Unit 4 notes](https://elevien.github.io/math50_sandbox/unit4/).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import norm

rng = np.random.default_rng(0)

### Example: two-predictor regression in `statsmodels`

In [2]:
# --- simulate data ---
n = 300
beta0, beta1, beta2, sigma = 1.0, 2.0, -1.0, 0.5
X1 = rng.normal(size=n)
X2 = rng.normal(size=n)
eps = rng.normal(scale=sigma, size=n)
Y = beta0 + beta1*X1 + beta2*X2 + eps

# --- fit OLS: Y ~ X1 + X2 ---
X = sm.add_constant(np.column_stack([X1, X2]))
results = sm.OLS(Y, X).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.959
Model:                            OLS   Adj. R-squared:                  0.959
Method:                 Least Squares   F-statistic:                     3457.
Date:                Thu, 27 Aug 2026   Prob (F-statistic):          1.99e-206
Time:                        12:30:18   Log-Likelihood:                -193.97
No. Observations:                 300   AIC:                             393.9
Df Residuals:                     297   BIC:                             405.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.9875      0.027     36.819      0.0

### Example: children's test scores

In [3]:
url = "https://raw.githubusercontent.com/avehtari/ROS-Examples/master/KidIQ/data/kidiq.csv"
df = pd.read_csv(url)

X = sm.add_constant(df[["mom_hs", "mom_iq"]])
fit2 = sm.OLS(df["kid_score"], X).fit()
print(fit2.summary())
print("R^2 =", fit2.rsquared)

# (d) P(mom_iq=90, no HS scores higher than mom_iq=110, HS)
sigma_hat = np.sqrt(fit2.mse_resid)
mean_A = fit2.params["const"] + fit2.params["mom_hs"]*0 + fit2.params["mom_iq"]*90
mean_B = fit2.params["const"] + fit2.params["mom_hs"]*1 + fit2.params["mom_iq"]*110
diff_sd = np.sqrt(2) * sigma_hat
p_A_beats_B = 1 - norm.cdf(0, loc=mean_A - mean_B, scale=diff_sd)
print("P(A does better than B) =", p_A_beats_B)

                            OLS Regression Results                            
Dep. Variable:              kid_score   R-squared:                       0.214
Model:                            OLS   Adj. R-squared:                  0.210
Method:                 Least Squares   F-statistic:                     58.72
Date:                Thu, 27 Aug 2026   Prob (F-statistic):           2.79e-23
Time:                        12:30:18   Log-Likelihood:                -1872.0
No. Observations:                 434   AIC:                             3750.
Df Residuals:                     431   BIC:                             3762.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         25.7315      5.875      4.380      0.0

### Example: test scores — multiple vs. single predictor

In [4]:
X1p = sm.add_constant(df[["mom_hs"]])
fit1 = sm.OLS(df["kid_score"], X1p).fit()

print("single-predictor beta_hs' =", fit1.params["mom_hs"])
print("two-predictor beta_hs     =", fit2.params["mom_hs"])

single-predictor beta_hs' = 11.77126099706744
two-predictor beta_hs     = 5.950116913726478


### Example: Simpson's paradox with two binary predictors

In [5]:
P = {(0, 0): 0.4, (0, 1): 0.1, (1, 0): 0.1, (1, 1): 0.4}
P_X2_1_given_X1_1 = P[(1, 1)] / (P[(1, 0)] + P[(1, 1)])
P_X2_1_given_X1_0 = P[(0, 1)] / (P[(0, 0)] + P[(0, 1)])
beta_12 = P_X2_1_given_X1_1 - P_X2_1_given_X1_0

beta1, beta2 = 1, -2
beta1_prime = beta1 + beta2 * beta_12
print("beta_1,2 =", beta_12, " beta1' =", beta1_prime, " (sign flip from beta1 =", beta1, ")")

# check by simulation
N = 500_000
keys = [(0, 0), (0, 1), (1, 0), (1, 1)]
idx = rng.choice(4, size=N, p=[P[k] for k in keys])
X1s = np.array([keys[i][0] for i in idx])
X2s = np.array([keys[i][1] for i in idx])
Ys = beta1 * X1s + beta2 * X2s + rng.normal(0, 0.1, size=N)
print("beta1' simulated:", np.cov(X1s, Ys)[0, 1] / np.var(X1s))

beta_1,2 = 0.6000000000000001  beta1' = -0.20000000000000018  (sign flip from beta1 = 1 )
beta1' simulated: -0.19715600407464323


### Example: correlated predictors

In [6]:
b, beta1, beta2, sigma = 0.7, 1.5, -0.5, 1.0
N = 200_000
X1 = rng.normal(size=N)
X2 = b * X1 + np.sqrt(1 - b**2) * rng.normal(size=N)
Y = beta1 * X1 + beta2 * X2 + rng.normal(scale=sigma, size=N)

print("cov(X1,X2) ~", np.cov(X1, X2)[0, 1], " (theory b =", b, ")")

beta1_hat = (np.cov(X1, Y)[0, 1] - np.cov(X2, Y)[0, 1] * b) / (1 - b**2)
print("beta1 via covariance formula:", beta1_hat, " true:", beta1)

beta1_prime_emp = np.cov(X1, Y)[0, 1] / np.var(X1)
print("single-predictor beta1' empirical:", beta1_prime_emp, " theory beta1+b*beta2:", beta1 + b * beta2)

cov(X1,X2) ~ 0.6983459990242273  (theory b = 0.7 )
beta1 via covariance formula: 1.4882754174787052  true: 1.5
single-predictor beta1' empirical: 1.14797145254497  theory beta1+b*beta2: 1.15


### Example: racial disparities in earnings

In [7]:
url = "https://raw.githubusercontent.com/avehtari/ROS-Examples/master/Earnings/data/earnings.csv"
edf = pd.read_csv(url)

dummies = pd.get_dummies(edf["ethnicity"], drop_first=True, dtype=float)  # drops Black (reference)
X = sm.add_constant(dummies)
fit = sm.OLS(edf["earnk"], X).fit()
print(fit.summary())

gap = fit.params["White"] - fit.params["Hispanic"]
print("Estimated White - Hispanic earnings gap ($1000s):", gap)

                            OLS Regression Results                            
Dep. Variable:                  earnk   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     4.602
Date:                Thu, 27 Aug 2026   Prob (F-statistic):            0.00325
Time:                        12:30:18   Log-Likelihood:                -8226.1
No. Observations:                1816   AIC:                         1.646e+04
Df Residuals:                    1812   BIC:                         1.648e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         17.1384      1.674     10.235      0.0